# Auditoria de Calidad  -  Notebooks del Curso

Este notebook audita automaticamente todos los `B*.ipynb` del directorio.  
Para cada notebook ejecuta **11 checks** y produce un reporte **OK / WARN / FAIL**.  

| Estado | Significado | Accion |
|---|---|---|
| OK   | Check superado sin observaciones | Ninguna |
| WARN | Problema menor o decision no documentada | Anadir comentario explicativo |
| FAIL | Error que invalida el resultado o la reproducibilidad | Corregir antes de usar |
| SKIP | Check no aplica a este notebook | Ninguna |

> Ejecutar con: `Kernel -> Restart & Run All`  
> Resultado guardado en `audit_report.csv`

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Importar modulo de auditoria (debe estar en el mismo directorio)
import notebook_audit_utils as nau

print("notebook_audit_utils cargado [OK]")
print(f"Python: {sys.version.split()[0]}")
print(f"pandas: {pd.__version__} | numpy: {np.__version__}")

## 1. Descubrimiento de notebooks

Lista de todos los notebooks `B*.ipynb` encontrados en el directorio actual.

In [ ]:
nb_paths = sorted(Path(".").glob("B*.ipynb"))
print(f"Notebooks encontrados: {len(nb_paths)}\n")
for p in nb_paths:
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name:<55} {size_kb:>7.1f} KB")

## 2. Ejecucion de la auditoria

Se aplican los 11 checks a cada notebook.  
La funcion `audit_all_notebooks()` retorna un DataFrame con todas las filas  
(una por notebook x check) y una lista de `AuditReport` individuales.

In [ ]:
df_results, reports = nau.audit_all_notebooks(
    directory=".",
    pattern="B*.ipynb",
    verbose=False  # True para ver detalle de cada notebook
)

print(f"Total de checks ejecutados: {len(df_results)}")
print(f"Notebooks auditados: {df_results.notebook.nunique()}")
print()
totales = df_results["status"].value_counts()
for estado in ["FAIL", "WARN", "OK", "SKIP"]:
    n = totales.get(estado, 0)
    print(f"  {estado:<5}: {n:>3} ({n/len(df_results)*100:.1f}%)")

## 3. Tabla resumen por notebook

Cada fila = un notebook. Columnas = conteo de checks por estado.

In [ ]:
pivot = (
    df_results
    .groupby(["notebook", "status"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["FAIL", "WARN", "OK", "SKIP"], fill_value=0)
)
pivot["TOTAL"] = pivot.sum(axis=1)
pivot["SCORE"] = pivot["FAIL"] * -10 + pivot["WARN"] * -1 + pivot["OK"] * 2
pivot = pivot.sort_values("SCORE")

print("Resumen por notebook (ordenado de mayor a menor gravedad):\n")
print(pivot.to_string())
print()
n_fail_nb = (pivot["FAIL"] > 0).sum()
n_warn_nb = ((pivot["FAIL"] == 0) & (pivot["WARN"] > 0)).sum()
n_ok_nb   = ((pivot["FAIL"] == 0) & (pivot["WARN"] == 0)).sum()
print(f"Notebooks con FAIL: {n_fail_nb}")
print(f"Notebooks con WARN (sin FAIL): {n_warn_nb}")
print(f"Notebooks limpios: {n_ok_nb}")

## 4. Fig01: Heatmap notebook x check

Verde = OK | Gris = SKIP | Naranja = WARN | Rojo = FAIL

In [ ]:
n_nb = df_results["notebook"].nunique()
fig_h = max(5, n_nb * 0.55 + 3)
fig, ax = plt.subplots(figsize=(16, fig_h))
nau.plot_audit_heatmap(
    df_results, ax=ax,
    title="Auditoria de calidad  -  todos los notebooks del curso\n(11 checks por notebook)"
)
plt.tight_layout()
plt.savefig("images/ZZ_AUDIT_fig01.png", bbox_inches="tight", dpi=150)
plt.show()
print("fig01 guardada")

## 5. Fig02: Resumen FAIL/WARN/OK por notebook

Los notebooks con mas problemas aparecen arriba.

In [ ]:
fig, ax = plt.subplots(figsize=(10, max(5, n_nb * 0.5 + 2)))
nau.plot_audit_bars(df_results, ax=ax)
plt.tight_layout()
plt.savefig("images/ZZ_AUDIT_fig02.png", bbox_inches="tight", dpi=150)
plt.show()
print("fig02 guardada")

## 6. Detalle de hallazgos FAIL

Los checks con estado FAIL deben corregirse antes de usar el notebook en formacion.

In [ ]:
df_fail = df_results[df_results["status"] == "FAIL"]
if df_fail.empty:
    print("[OK] No hay checks FAIL en ningun notebook.")
else:
    print(f"[FAIL] {len(df_fail)} checks FAIL encontrados:\n")
    for _, row in df_fail.iterrows():
        print(f"  {row.notebook:<50} {row.check:<30} -> {row.detail}")

## 7. Detalle de hallazgos WARN

Los WARN son oportunidades de mejora o decisiones que necesitan documentarse.

In [ ]:
df_warn = df_results[df_results["status"] == "WARN"]
if df_warn.empty:
    print("[OK] No hay checks WARN en ningun notebook.")
else:
    print(f"[WARN] {len(df_warn)} checks WARN encontrados:\n")
    # Agrupar por check para ver patrones transversales
    por_check = df_warn.groupby("check")["notebook"].apply(list)
    for check_name, nbs in por_check.items():
        print(f"  {check_name}")
        for nb in nbs:
            detail = df_warn[
                (df_warn.notebook == nb) & (df_warn.check == check_name)
            ]["detail"].values[0]
            print(f"    -> {nb}: {detail}")
        print()

## 8. Checks con mayor tasa de WARN/FAIL (patron transversal)

Estos checks identifican problemas sistematicos aplicables a toda la coleccion.

In [ ]:
por_check_estado = (
    df_results[df_results["status"].isin(["FAIL", "WARN"])]
    .groupby(["check", "status"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["FAIL", "WARN"], fill_value=0)
)
por_check_estado["TOTAL_PROBLEMAS"] = por_check_estado.sum(axis=1)
por_check_estado = por_check_estado.sort_values("TOTAL_PROBLEMAS", ascending=False)

print("Checks con mas problemas transversales:\n")
print(por_check_estado[por_check_estado["TOTAL_PROBLEMAS"] > 0].to_string())
print()
check_mas_critico = por_check_estado.index[0] if len(por_check_estado) > 0 else "ninguno"
print(f"Check mas problematico: {check_mas_critico}")

## 9. Exportar reporte completo

`audit_report.csv` contiene todos los checks con su estado y detalle,  
listo para compartir con el equipo o importar en una hoja de calculo.

In [ ]:
# Guardar CSV completo
report_path = Path("audit_report.csv")
df_results.to_csv(report_path, index=False, encoding="utf-8")
print(f"Reporte guardado en: {report_path.resolve()}")
print(f"Filas: {len(df_results)} | Columnas: {list(df_results.columns)}")

# Guardar resumen ejecutivo
pivot_path = Path("audit_summary.csv")
pivot.to_csv(pivot_path, encoding="utf-8")
print(f"Resumen guardado en: {pivot_path.resolve()}")

## 10. Como usar notebook_audit_utils en cualquier notebook del curso

```python
# Al inicio del notebook a revisar:
import notebook_audit_utils as nau

# Despues de cargar el DataFrame:
nau.assert_shape(df, expected_rows=200, expected_cols=12, name="df_proyectos")
nau.assert_no_nulls(df, cols=["retraso_proyecto", "tickets_criticos"])
nau.assert_range(df["margen_anual"], 0.0, 1.0, name="margen_anual")

# Despues del split:
nau.assert_no_overlap(df_train, df_test, id_col="cliente_id")

# Para auditar un notebook especifico:
report = nau.audit_notebook("B12D_riesgo_proyectos.ipynb", verbose=True)
```

**Patrones a evitar:**

```python
# MAL: fit sobre todos los datos antes del split
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # leakage si luego hay split
X_train, X_test = train_test_split(X_scaled)

# BIEN: Pipeline protege contra leakage
from sklearn.pipeline import Pipeline
pipe = Pipeline([("scaler", StandardScaler()), ("model", RandomForestClassifier())])
pipe.fit(X_train, y_train)  # scaler solo ve X_train
```

## Conclusion

La auditoria automatica detecta problemas reproducibles y transversales.  
No reemplaza la revision humana, pero prioriza donde mirar primero.

**Protocolo de accion recomendado:**

1. Corregir todos los FAIL antes de usar el notebook en formacion
2. Documentar los WARN que sean decisiones intencionales con un comentario
3. Re-ejecutar `Kernel -> Restart & Run All` en cada notebook corregido
4. Re-ejecutar este notebook para verificar que los FAIL desaparecieron

---
*Generado con `notebook_audit_utils.py`  -  auditoria de calidad del curso de formacion IA.*